In [22]:
#### Core script to defines and execute rf runs using functions defined in the following scripts

# config          # configures the model run with input data and defaults (where not directly input)
# spatial_cv      # set up the spatial CV blocks based on fold CSV
# train           # trains model and predicts on test data
# evaluate        # calculates metrics for each fold and overall 
# importance      # calculates feature importance + permutation importance for each fold and overall 

from pathlib import Path
import json
import dataclasses
import pandas as pd

from step_a_config import RunConfig
from step_b_spatial_cv import make_spatial_folds
from step_c_train import train_model
from step_d_evaluate import evaluate
from step_e_importance import get_feature_importance

In [23]:
##### Set-up (doesn't change)

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/") 

In [24]:
#### DEFINE RUN (MANUAL)

version = 'labor' # capital or labor
unit = 'tonne' # USD or tonne
variable_def = 'rtv' # abs or rtv
type_of_model = 'rf' # rf or qrf

name_of_run =  f'{version}_{unit}_{variable_def}_{type_of_model}' 

In [25]:
##### Configuration (some manual changes)

# set target variable 
unit_map = {
    ("capital", "tonne"): "tonne",
    ("capital", "USD"): "USD",
    ("labor", "tonne"): "tonne",
    ("labor", "USD"): "million_USD",
}
unit_final = unit_map[(version, unit)]

numerator = "USD" if version == "capital" else "jobs"
country_labor_unit = "million_USD" if unit == "USD" else "tonne"
precedent = "rtv_" if variable_def == "rtv" else ""

target_var = f'{precedent}log_{version}_intensity_{numerator}_per_{unit_final}' 
 
# turn on if running absolute model 
# country_capital_var = f'log_country_capital_intensity_USD_per_{unit}' 
# country_labor_var = f'log_country_labor_intensity_jobs_per_{country_labor_unit}' 

# config
model_data_type = "absolute" if variable_def == "abs" else "relative"

fold = FOLD_DIR / f"{version}_folds.csv"
model_data = f"{version}_{model_data_type}_final.csv"

RUNS = [
    RunConfig(
        run_name         = name_of_run,
        target           = target_var,   
        dataset          = model_data,
        fold_assignments = fold,
        model_type       = type_of_model,
        id_cols          = ["PROJ_ID", "country_ID"],
    ),
]

# define which columns are feature columns 

# absolute model 
# feature_cols = ['SOC', 'pct_cropland_irrigated', 'female_share',
#        'pop_density_people_per_km2', 'average_travel_time_city',
#        'average_travel_time_port', 'probability_economic_land_use_objective',
#        'probability_survival_land_use_objective', 'growing_season_length_days',
#        'child_dependency_ratio', 'GDP_pc', 'slope',
#        'cereals_share_production_USD', 'fibres_share_production_USD',
#        'fruits_share_production_USD', 'oilcrops_share_production_USD',
#        'pulses_share_production_USD', 'roots_tubers_share_production_USD',
#        'rest_of_crops_share_production_USD',
#        'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
#        'rubber_share_production_USD', 'ruminants_share_production_USD',
#        'monogastrics_share_production_USD', 'poultry_share_production_USD',
#        'pct_GDP_ag', 'share_large_field', 'share_medium_field',
#        'share_small_field', 'share_vsmall_field', 'lon', 'lat',
#        'share_with_nightlights', 'crop_intensity', 'log_tonnes_production_per_HA',
#        'log_USD_production_per_HA', 'buffalo_density_per_km2',
#        'chicken_density_per_km2', 'cattle_density_per_km2',
#        'goats_density_per_km2', 'pigs_density_per_km2',
#        'sheep_density_per_km2', 'livestock_density_LU_per_km2',
#        country_capital_var, country_labor_var]

feature_cols = ['rtv_log_pop_density_people_per_km2', 'rtv_log_USD_production_per_HA',
       'rtv_log_tonnes_production_per_HA', 'rtv_log_buffalo_density_per_km2',
       'rtv_log_chicken_density_per_km2', 'rtv_log_cattle_density_per_km2',
       'rtv_log_goats_density_per_km2', 'rtv_log_pigs_density_per_km2',
       'rtv_log_sheep_density_per_km2', 'rtv_log_livestock_density_LU_per_km2',
       'rtv_SOC', 'rtv_pct_cropland_irrigated', 'rtv_female_share',
       'rtv_average_travel_time_city', 'rtv_average_travel_time_port',
       'rtv_probability_economic_land_use_objective',
       'rtv_probability_survival_land_use_objective',
       'rtv_growing_season_length_days', 'rtv_child_dependency_ratio',
       'rtv_GDP_pc', 'rtv_slope', 'rtv_cereals_share_production_USD',
       'rtv_fibres_share_production_USD', 'rtv_fruits_share_production_USD',
       'rtv_oilcrops_share_production_USD', 'rtv_pulses_share_production_USD',
       'rtv_roots_tubers_share_production_USD',
       'rtv_rest_of_crops_share_production_USD',
       'rtv_sugar_crops_share_production_USD',
       'rtv_vegetables_share_production_USD',
       'rtv_rubber_share_production_USD', 'rtv_ruminants_share_production_USD',
       'rtv_monogastrics_share_production_USD',
       'rtv_poultry_share_production_USD', 'rtv_pct_GDP_ag',
       'rtv_share_large_field', 'rtv_share_medium_field',
       'rtv_share_small_field', 'rtv_share_vsmall_field',
       'rtv_share_with_nightlights', 'rtv_crop_intensity']


In [26]:
##### Define function to save results into defined directory
def save_results(results, config, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True) # creates folder if it doesn't already exist 

    # save config details for run 
    # convert any Path objects to strings so json can handle them
    config_dict = dataclasses.asdict(config)
    config_dict = {
        k: str(v) if isinstance(v, Path) else v
        for k, v in config_dict.items()
    }
    (out_dir / "config.json").write_text(
        json.dumps(config_dict, indent=2)
    )

    # save predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # save the best parameters for each fold
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # save metrics and importance scores
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["importance"].to_csv(out_dir / "importance.csv", index=False)

    print(f"  saved to {out_dir}")

In [27]:
##### RUN   

# define the function to run all scripts in order 
def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load model data
    df = pd.read_csv(DATA_DIR / config.dataset)

    # run spatial folds functions 
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # run training and prediction functions
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # run evaluation functions
    results["metrics"] = evaluate(results, config)

    # run importance functions
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # run save function
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)

# Run run function for each model configuration defined above
for config in RUNS:
    run(config)


════════════════════════════════════════════════════════════
  run: labor_tonne_rtv_rf
  target: rtv_log_labor_intensity_jobs_per_tonne  |  model: rf
════════════════════════════════════════════════════════════

── Spatial folds ────────────────────────────────────────
Excluded 6 rows from countries missing from fold CSV
  fold_1: 32 train countries (6,117 rows) | 1 test countries (5,375 rows)
  fold_2: 32 train countries (10,149 rows) | 1 test countries (1,343 rows)
  fold_3: 32 train countries (8,755 rows) | 1 test countries (2,737 rows)
  fold_4: 30 train countries (2,037 rows) | 3 test countries (9,455 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_2 ──────────────────────────────
  test countries: ['CAN']
  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



── Feature Importance (top 15) ──────────────────────────────
                              feature  impurity_mean  impurity_std  permutation_mean  permutation_std
     rtv_log_tonnes_production_per_HA       0.109145      0.047323          0.112133         0.052062
                rtv_share_large_field       0.065118      0.043017          0.089766         0.051671
   rtv_ruminants_share_production_USD       0.061299      0.052402          0.079893         0.033014
                   rtv_crop_intensity       0.106325      0.060072          0.042120         0.031453
 rtv_sugar_crops_share_production_USD       0.051681      0.041698          0.035269         0.020472
           rtv_pct_cropland_irrigated       0.039056      0.020313          0.032570         0.000843
      rtv_log_chicken_density_per_km2       0.030444      0.006310          0.028666         0.021025
    rtv_oilcrops_share_production_USD       0.063390      0.053103          0.024783         0.019648
 rtv_log_livestock_